# GRU-grid model across N and Omega
Train on `GenerateTraj/data*` and generate trajectories at seen or interpolated conditions.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == 'different_omega' else Path.cwd()
sys.path.insert(0, str(project_root))
from different_omega.dataset import MultiNOmegaNextStateDataset
from different_omega.model import ConditionedNOmegaTrajectoryGRUGrid
from different_omega.train_grid_model import load_trajectories, generate_trajectories

data_root = project_root / 'GenerateTraj'
checkpoint_path = project_root / 'different_omega' / 'different_omega_grugrid_best.pt'
n_values = [4, 6, 8, 10, 12, 14, 16, 18]
omega_values = None  # e.g. [.61, .65, .69, .71, .75]

## Inspect available conditions

In [ ]:
sz, values_n, values_omega, source_files = load_trajectories(data_root, n_values, omega_values, 5)
print('trajectories:', sz.shape)
print('N:', sorted(set(values_n.tolist())))
print('Omega/Omega_c:', sorted(set(values_omega.tolist())))

## Train
Run this cell after choosing the data volume and hyperparameters. Missing N folders are skipped automatically.

In [ ]:
!python -m different_omega.train_grid_model --data-root GenerateTraj --n-values 4 6 8 10 12 14 16 18 --trajectories-per-condition 500 --epochs 100 --checkpoint different_omega/different_omega_grugrid_best.pt

## Generate at different N and Omega

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model = ConditionedNOmegaTrajectoryGRUGrid(**checkpoint['model_config']).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

conditions = [(8, .61), (8, .68), (12, .71), (16, .75)]
initial_sz = torch.full((32,), -1.0)
generated = {condition: generate_trajectories(model, condition[0], condition[1], initial_sz, sz.shape[1], time_end=checkpoint['time_end'], temperature=.8) for condition in conditions}
print({condition: values.shape for condition, values in generated.items()})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for axis, (condition, values) in zip(axes.flat, generated.items()):
    axis.plot(values[:8].cpu().T, alpha=.35)
    axis.plot(values.mean(0).cpu(), color='black', linewidth=2)
    axis.set_title(f'N={condition[0]}, Omega/Omega_c={condition[1]:.2f}')
    axis.set_xlabel('normalized time')
    axis.set_ylabel('s_z')
plt.tight_layout()